# 06 — Outlier Rejection, Sub-pixel Refinement & Warping

Takes the raw match CSVs from Notebook 05 (now **SIFT, SuperPoint+LightGlue,
and RoMa**) and:
1. Rescales the "fine" side of each match back to full resolution (it was downsampled in Notebook 04).
2. Fits a robust geometric transform (MAGSAC, falling back to RANSAC) per method and keeps whichever gives the most inliers, and rejects outliers.
3. Enforces uniform spatial coverage of matches (a stated requirement).
4. Refines surviving matches to sub-pixel accuracy via local phase correlation — using **windowed reads**, not full-image loads, to stay memory-safe.
5. Warps the source image onto the reference grid via **`gdalwarp`** (not in-Python `cv2.warpPerspective`) — GDAL warps block-by-block internally, which is the memory-safe way to warp a full-resolution OHRC strip.

**Output:**
```
26166_DATA/processed/registered/{pair}/registered.tif        <- the deliverable "registered product"
26166_DATA/processed/registered/{pair}/match_points.csv       <- the deliverable "match points"
26166_DATA/processed/registered/{pair}/transform_meta.json    <- inlier count/ratio, method used
```


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, csv, json, subprocess
import numpy as np
import cv2
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt

DRIVE_ROOT = '/content/drive/MyDrive/26166_DATA'
MATCH_ROOT = f'{DRIVE_ROOT}/processed/matches'
OUT_ROOT   = f'{DRIVE_ROOT}/processed/registered'
os.makedirs(OUT_ROOT, exist_ok=True)

with open(f'{DRIVE_ROOT}/processed/manifest_scalespace.csv') as f:
    scalespace_manifest = list(csv.DictReader(f))
with open(f'{DRIVE_ROOT}/processed/manifest_radiometric.csv') as f:
    radio_rows = list(csv.DictReader(f))

def get_product_path(instrument):
    match = next(r for r in radio_rows if r['instrument'] == instrument)
    return match['radiometric_path']


Mounted at /content/drive


## 1. Load raw matches and rescale the fine side back to full resolution

In [2]:
def load_raw_matches(pair, method):
    path = f'{MATCH_ROOT}/{pair}/{method}_raw_matches.csv'
    if not os.path.exists(path):
        return np.empty((0, 2)), np.empty((0, 2))
    pts1, pts2 = [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            pts1.append([float(row['x_fine']), float(row['y_fine'])])
            pts2.append([float(row['x_coarse']), float(row['y_coarse'])])
    return np.array(pts1), np.array(pts2)

def rescale_fine_points(pts_fine_scaled, pyramid_level):
    factor = 2 ** int(pyramid_level)
    return pts_fine_scaled * factor


## 2. Robust transform fitting (MAGSAC → RANSAC fallback) + train/test split

We hold out 20% of candidate matches *before* fitting, purely so Notebook 07
can report a genuine RMSE on points the transform never saw.


In [3]:
def fit_robust_transform(pts_src, pts_dst, model='homography', thresh=3.0, test_frac=0.2, seed=42):
    n = len(pts_src)
    if n < 8:
        return None

    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_test = max(4, int(n * test_frac))
    test_idx, train_idx = idx[:n_test], idx[n_test:]

    src_train, dst_train = pts_src[train_idx], pts_dst[train_idx]
    src_test,  dst_test  = pts_src[test_idx],  pts_dst[test_idx]

    method_flag = getattr(cv2, 'USAC_MAGSAC', cv2.RANSAC)
    method_name = 'MAGSAC' if method_flag != cv2.RANSAC else 'RANSAC'

    if model == 'homography':
        M, mask = cv2.findHomography(src_train, dst_train, method_flag, thresh)
    else:
        M, mask = cv2.estimateAffinePartial2D(src_train, dst_train, method=method_flag,
                                               ransacReprojThreshold=thresh)
    if M is None:
        return None

    mask = mask.ravel().astype(bool)
    return {
        'M': M, 'model': model, 'method_name': method_name,
        'train_src': src_train, 'train_dst': dst_train, 'train_inlier_mask': mask,
        'test_src': src_test, 'test_dst': dst_test,
    }

def apply_transform(M, pts, model):
    if model == 'homography':
        pts_h = cv2.perspectiveTransform(pts.reshape(-1, 1, 2).astype(np.float32), M)
        return pts_h.reshape(-1, 2)
    else:
        pts_h = cv2.transform(pts.reshape(-1, 1, 2).astype(np.float32), M)
        return pts_h.reshape(-1, 2)


## 3. Uniform-coverage enforcement (grid-based)

In [4]:
def enforce_uniform_coverage(pts_ref, inlier_mask, ref_shape, grid=8, max_per_cell=None):
    h, w = ref_shape
    cell_h, cell_w = h / grid, w / grid
    inlier_idx = np.where(inlier_mask)[0]

    cells_map = {}
    for i in inlier_idx:
        x, y = pts_ref[i]
        cx, cy = min(int(x // cell_w), grid - 1), min(int(y // cell_h), grid - 1)
        cells_map.setdefault((cx, cy), []).append(i)

    coverage_score = len(cells_map) / (grid * grid)

    kept_idx = inlier_idx
    if max_per_cell is not None:
        kept_idx = []
        for cell_idxs in cells_map.values():
            kept_idx.extend(cell_idxs[:max_per_cell])
        kept_idx = np.array(sorted(kept_idx))

    final_mask = np.zeros_like(inlier_mask)
    final_mask[kept_idx] = True
    return final_mask, coverage_score


## 4. Sub-pixel refinement via windowed phase correlation

Reads small patches from disk with `rasterio.windows.Window` instead of the
full raster — memory-safe, avoids full-image RAM spikes.


In [5]:
def read_patch(path, cx, cy, half_size=32):
    with rasterio.open(path) as src:
        h, w = src.height, src.width
        x0 = int(max(0, min(cx - half_size, w - 2 * half_size)))
        y0 = int(max(0, min(cy - half_size, h - 2 * half_size)))
        window = Window(x0, y0, 2 * half_size, 2 * half_size)
        patch = src.read(1, window=window)
    return patch.astype(np.float32), (x0, y0)

def refine_subpixel(pts_src_full, pts_dst_full, src_path, dst_path, half_size=32):
    refined = pts_dst_full.copy()
    shifts = []
    for i in range(len(pts_dst_full)):
        sx, sy = pts_dst_full[i]
        dx, dy = pts_dst_full[i]
        try:
            patch_src, _ = read_patch(dst_path, sx, sy, half_size)
            patch_dst, _ = read_patch(dst_path, dx, dy, half_size)
            (shift_x, shift_y), _ = cv2.phaseCorrelate(
                np.float32(patch_src) - patch_src.mean(),
                np.float32(patch_dst) - patch_dst.mean())
            refined[i] = [dx + shift_x, dy + shift_y]
            shifts.append((shift_x, shift_y))
        except Exception:
            shifts.append((0.0, 0.0))
    return refined, np.array(shifts)


## 5. Run the full stage for every pair × method, keep the best

In [6]:
results = {}

for m in scalespace_manifest:
    pair = m['pair']
    level = m['pyramid_level_applied']
    fine_instrument, coarse_instrument = m['fine_instrument'], m['coarse_instrument']
    fine_full_path = get_product_path(fine_instrument)
    coarse_full_path = get_product_path(coarse_instrument)

    with rasterio.open(coarse_full_path) as src:
        ref_h, ref_w = src.height, src.width

    best = None
    for method in ['sift', 'lightglue', 'roma']:
        pts_fine_scaled, pts_coarse = load_raw_matches(pair, method)
        if len(pts_fine_scaled) < 8:
            print(f'{pair} [{method}]: too few raw matches, skipping')
            continue
        pts_fine_full = rescale_fine_points(pts_fine_scaled, level)

        fit = fit_robust_transform(pts_fine_full, pts_coarse, model='homography')
        if fit is None:
            print(f'{pair} [{method}]: transform fitting failed')
            continue

        inlier_ratio = fit['train_inlier_mask'].mean()
        n_inliers = fit['train_inlier_mask'].sum()
        print(f"{pair} [{method}]: {fit['method_name']}, {n_inliers} inliers / "
              f"{len(fit['train_inlier_mask'])} candidates (ratio {inlier_ratio:.2f})")

        if best is None or n_inliers > best['n_inliers']:
            best = {'method': method, 'fit': fit, 'n_inliers': n_inliers, 'inlier_ratio': inlier_ratio}

    if best is None:
        print(f'{pair}: NO usable matches from any method — skipping this pair')
        continue

    fit = best['fit']
    coverage_mask, coverage_score = enforce_uniform_coverage(
        fit['train_dst'], fit['train_inlier_mask'], (ref_h, ref_w), grid=8, max_per_cell=15)

    results[pair] = {**best, 'coverage_mask': coverage_mask, 'coverage_score': coverage_score,
                      'fine_full_path': fine_full_path, 'coarse_full_path': coarse_full_path,
                      'fine_instrument': fine_instrument, 'coarse_instrument': coarse_instrument}
    print(f'{pair}: chosen method = {best["method"]}, coverage score = {coverage_score:.2f}')


/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


ohrc_to_tmc2 [sift]: MAGSAC, 4 inliers / 32 candidates (ratio 0.12)
ohrc_to_tmc2 [lightglue]: too few raw matches, skipping
ohrc_to_tmc2 [roma]: MAGSAC, 7 inliers / 4000 candidates (ratio 0.00)
ohrc_to_tmc2: chosen method = roma, coverage score = 0.06
tmc2_to_iirs [sift]: too few raw matches, skipping
tmc2_to_iirs [lightglue]: too few raw matches, skipping
tmc2_to_iirs [roma]: MAGSAC, 97 inliers / 4000 candidates (ratio 0.02)
tmc2_to_iirs: chosen method = roma, coverage score = 0.06


## 6. Sub-pixel refine the kept inliers, then save the match-points deliverable

In [7]:
for pair, r in results.items():
    fit = r['fit']
    kept = r['coverage_mask']
    src_pts = fit['train_src'][kept]
    dst_pts = fit['train_dst'][kept]

    refined_dst, shifts = refine_subpixel(src_pts, dst_pts, r['fine_full_path'], r['coarse_full_path'])

    out_dir = f'{OUT_ROOT}/{pair}'
    os.makedirs(out_dir, exist_ok=True)
    match_csv = f'{out_dir}/match_points.csv'
    with open(match_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['x_src', 'y_src', 'x_dst_raw', 'y_dst_raw', 'x_dst_subpixel', 'y_dst_subpixel'])
        for i in range(len(src_pts)):
            writer.writerow([src_pts[i][0], src_pts[i][1], dst_pts[i][0], dst_pts[i][1],
                              refined_dst[i][0], refined_dst[i][1]])

    meta = {
        'pair': pair, 'method': r['method'], 'model': fit['model'],
        'ransac_variant': fit['method_name'], 'n_inliers_kept': int(kept.sum()),
        'inlier_ratio_before_coverage_filter': float(r['inlier_ratio']),
        'coverage_score': float(r['coverage_score']),
        'mean_subpixel_shift_px': float(np.mean(np.linalg.norm(shifts, axis=1))) if len(shifts) else None,
    }
    with open(f'{out_dir}/transform_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    # saved for Notebook 07's held-out RMSE calculation
    np.save(f'{out_dir}/M.npy', fit['M'])
    np.savez(f'{out_dir}/test_points.npz', src=fit['test_src'], dst=fit['test_dst'])

    print(f'{pair}: saved {kept.sum()} refined match points -> {match_csv}')
    print(f'         mean sub-pixel correction: {meta["mean_subpixel_shift_px"]:.3f} px')


ohrc_to_tmc2: saved 7 refined match points -> /content/drive/MyDrive/26166_DATA/processed/registered/ohrc_to_tmc2/match_points.csv
         mean sub-pixel correction: 0.000 px
tmc2_to_iirs: saved 35 refined match points -> /content/drive/MyDrive/26166_DATA/processed/registered/tmc2_to_iirs/match_points.csv
         mean sub-pixel correction: 0.000 px


## 7. Warp the source onto the reference grid via `gdalwarp` (memory-safe)

We write the kept, refined matches as GDAL ground-control-points on the
**source** (fine) image, then let `gdalwarp` do the actual resampling —
avoids holding a full-resolution OHRC array in Python memory.


In [8]:
def warp_with_gdal(src_path, dst_shape_path, match_csv_path, out_path, order=1):
    gcp_args = []
    with open(match_csv_path) as f:
        for row in csv.DictReader(f):
            x_src, y_src = row['x_src'], row['y_src']
            x_dst, y_dst = row['x_dst_subpixel'], row['y_dst_subpixel']
            gcp_args += ['-gcp', x_src, y_src, x_dst, y_dst]

    tmp_gcp_tif = src_path.replace('.tif', '_withgcps.tif')
    subprocess.run(['gdal_translate'] + gcp_args + [src_path, tmp_gcp_tif], check=True)

    with rasterio.open(dst_shape_path) as ref:
        ref_w, ref_h = ref.width, ref.height

    subprocess.run([
        'gdalwarp', '-r', 'bilinear', f'-order', str(order),
        '-ts', str(ref_w), str(ref_h),
        '-overwrite', tmp_gcp_tif, out_path
    ], check=True)
    return out_path

for pair, r in results.items():
    out_dir = f'{OUT_ROOT}/{pair}'
    match_csv = f'{out_dir}/match_points.csv'
    registered_path = f'{out_dir}/registered.tif'
    try:
        warp_with_gdal(r['fine_full_path'], r['coarse_full_path'], match_csv, registered_path, order=1)
        print(f'{pair}: registered product written -> {registered_path}')
    except subprocess.CalledProcessError as e:
        print(f'{pair}: gdalwarp failed ({e}) — try order=2 or check GCP spread across the image')


/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


ohrc_to_tmc2: registered product written -> /content/drive/MyDrive/26166_DATA/processed/registered/ohrc_to_tmc2/registered.tif


/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


tmc2_to_iirs: registered product written -> /content/drive/MyDrive/26166_DATA/processed/registered/tmc2_to_iirs/registered.tif


---
### Notes
- If `gdalwarp` complains about too few/collinear GCPs for the chosen
  `order`, loosen `thresh` in `fit_robust_transform` or raise `max_per_cell`.
- `order=1` is closest to a pure affine fit; bump to `order=2` if you have
  ≥10 well-spread GCPs and Notebook 07's RMSE looks high with order 1.
- `M.npy` / `test_points.npz` are now saved automatically in Section 6, so
  Notebook 07's RMSE cell will work without any extra edits.
- Next notebook: **07_evaluate.ipynb**
